# Retrieval Deep Dive

How different retrieval strategies find relevant documents — and when each one shines or fails.

**What we'll cover:**
1. Building a test corpus and indexing it
2. Dense retrieval (embedding similarity)
3. Sparse retrieval (BM25 keyword matching)
4. Hybrid retrieval (RRF and weighted fusion)
5. Cross-encoder reranking
6. Head-to-head comparison across query types
7. Latency benchmarks

In [ ]:
import sys
sys.path.insert(0, "..")

import time
import uuid

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi

from src.ingest.embedders import Embedder
from src.retrieve.hybrid_retriever import HybridRetriever
from src.retrieve.reranker import Reranker
from src.retrieve.retriever import Retriever
from src.store.chroma_store import ChromaStore

## 1. Build a Test Corpus

A small corpus with diverse topics so we can see how each retrieval method handles different query types: semantic, keyword-exact, and mixed.

In [ ]:
CORPUS = [
    # Programming
    "Python is a high-level programming language known for readability.",
    "The GIL (Global Interpreter Lock) limits Python's multi-threading performance.",
    "List comprehensions provide a concise way to create lists in Python.",
    # Machine Learning
    "Gradient descent optimizes neural network parameters by minimizing the loss function.",
    "The BERT-base model has 110M parameters and uses transformer architecture.",
    "Overfitting occurs when a model performs well on training data but poorly on unseen data.",
    # Geography
    "The Great Wall of China stretches over 13,000 miles across northern China.",
    "Mount Everest stands at 8,849 meters above sea level.",
    "The Amazon River carries more water than any other river in the world.",
    # Cooking
    "Sourdough bread requires a starter culture of wild yeast and lactobacilli.",
    "The Maillard reaction creates flavor when proteins and sugars are heated above 140C.",
    "Mise en place means preparing and organizing all ingredients before cooking.",
    # Science
    "The speed of light in a vacuum is approximately 299,792,458 meters per second.",
    "CRISPR-Cas9 enables precise editing of DNA sequences in living organisms.",
    "The half-life of Carbon-14 is approximately 5,730 years.",
]

embedder = Embedder("all-MiniLM-L6-v2")
store = ChromaStore(collection_name=f"nb03_{uuid.uuid4().hex[:8]}")
embeddings = embedder.embed(CORPUS)
store.add(texts=CORPUS, embeddings=embeddings)

print(f"Corpus: {len(CORPUS)} documents indexed")
print(f"Embedding dimension: {embedder.dimension}")

## 2. Dense Retrieval (Embedding Similarity)

The retriever embeds the query and finds the closest vectors in the store. Great for semantic understanding, but can miss exact keyword matches.

In [ ]:
retriever = Retriever(store, embedder)

test_queries = [
    ("Semantic", "How do neural networks learn?"),
    ("Keyword-exact", "What is CRISPR-Cas9?"),
    ("Mixed", "Python threading limitations"),
]

print("=== Dense Retrieval (top-3) ===\n")
for query_type, query in test_queries:
    results = retriever.retrieve(query, top_k=3)
    print(f"[{query_type}] Query: {query!r}")
    for i, r in enumerate(results):
        print(f"  #{i+1} (dist={r['distance']:.3f}) {r['text'][:80]}...")
    print()

## 3. Sparse Retrieval (BM25)

BM25 is a keyword-based ranking algorithm. It excels at exact term matches — acronyms, proper nouns, technical codes — that embedding models sometimes miss.

In [ ]:
tokenized_corpus = [doc.lower().split() for doc in CORPUS]
bm25 = BM25Okapi(tokenized_corpus)

print("=== BM25 Retrieval (top-3) ===\n")
for query_type, query in test_queries:
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_idx = scores.argsort()[::-1][:3]
    print(f"[{query_type}] Query: {query!r}")
    for rank, idx in enumerate(top_idx):
        if scores[idx] > 0:
            print(f"  #{rank+1} (score={scores[idx]:.3f}) {CORPUS[idx][:80]}...")
        else:
            print(f"  #{rank+1} (score=0.000) -- no keyword match --")
    print()

Notice how BM25 catches "CRISPR-Cas9" precisely (exact keyword match), while dense retrieval relies on semantic proximity. Conversely, BM25 struggles with paraphrased queries where no keywords overlap.

## 4. Hybrid Retrieval — Reciprocal Rank Fusion (RRF)

Hybrid retrieval combines both approaches. RRF merges rankings without needing score normalization:
`rrf_score = Σ 1/(k + rank)` for each ranker.

In [ ]:
hybrid = HybridRetriever(store, embedder, CORPUS)

print("=== Hybrid RRF Retrieval (top-3) ===\n")
for query_type, query in test_queries:
    results = hybrid.retrieve(query, top_k=3, fusion="rrf")
    print(f"[{query_type}] Query: {query!r}")
    for i, r in enumerate(results):
        print(f"  #{i+1} (rrf={r['score']:.4f}) {r['text'][:80]}...")
    print()

print("=== Hybrid Weighted Retrieval (0.7 dense, 0.3 BM25) ===\n")
for query_type, query in test_queries:
    results = hybrid.retrieve(query, top_k=3, fusion="weighted", dense_weight=0.7)
    print(f"[{query_type}] Query: {query!r}")
    for i, r in enumerate(results):
        print(f"  #{i+1} (score={r['score']:.4f}) {r['text'][:80]}...")
    print()

## 5. Cross-Encoder Reranking

A cross-encoder jointly scores (query, document) pairs — more accurate than bi-encoder similarity but slower (100-300ms). We retrieve broadly, then rerank to get the best results.

In [ ]:
reranker = Reranker("cross-encoder/ms-marco-MiniLM-L-2-v2")

query = "How do neural networks learn?"
initial = retriever.retrieve(query, top_k=10)
reranked = reranker.rerank(query, initial, top_k=5)

print(f"Query: {query!r}\n")
print("Before reranking (dense top-5):")
for i, r in enumerate(initial[:5]):
    print(f"  #{i+1} (dist={r['distance']:.3f}) {r['text'][:75]}...")

print("\nAfter reranking (top-5):")
for i, r in enumerate(reranked):
    print(f"  #{i+1} (rerank={r['rerank_score']:.3f}) {r['text'][:75]}...")

## 6. Head-to-Head Comparison

Let's run all methods against a broader set of queries and see which method retrieves the most relevant document at rank 1.

In [ ]:
eval_queries = [
    ("How do neural networks learn?", "Gradient descent"),
    ("What is CRISPR-Cas9?", "CRISPR-Cas9"),
    ("Python threading limitations", "GIL"),
    ("How fast does light travel?", "speed of light"),
    ("tallest mountain in the world", "Everest"),
    ("making sourdough bread", "Sourdough"),
    ("What is the Maillard reaction?", "Maillard"),
    ("Carbon dating half life", "Carbon-14"),
    ("BERT model architecture", "BERT-base"),
    ("largest river by volume", "Amazon"),
]

def get_rank1_text(method_name, query):
    if method_name == "Dense":
        return retriever.retrieve(query, top_k=1)[0]["text"]
    elif method_name == "BM25":
        tokens = query.lower().split()
        scores = bm25.get_scores(tokens)
        idx = scores.argmax()
        return CORPUS[idx] if scores[idx] > 0 else ""
    elif method_name == "Hybrid RRF":
        results = hybrid.retrieve(query, top_k=1, fusion="rrf")
        return results[0]["text"] if results else ""
    elif method_name == "Reranked":
        initial = retriever.retrieve(query, top_k=10)
        reranked = reranker.rerank(query, initial, top_k=1)
        return reranked[0]["text"] if reranked else ""

methods = ["Dense", "BM25", "Hybrid RRF", "Reranked"]
rows = []
for query, expected_keyword in eval_queries:
    row = {"Query": query[:35] + "...", "Expected": expected_keyword}
    for method in methods:
        text = get_rank1_text(method, query)
        hit = expected_keyword.lower() in text.lower()
        row[method] = "Hit" if hit else "Miss"
    rows.append(row)

df = pd.DataFrame(rows)

# Count hits per method
print("=== Rank-1 Hit Rate ===")
for method in methods:
    hits = sum(1 for r in rows if r[method] == "Hit")
    print(f"  {method:>12}: {hits}/{len(eval_queries)} ({100*hits/len(eval_queries):.0f}%)")
print()
df

## 7. Latency Benchmarks

How long does each method take per query? The reranker adds latency but improves quality.

In [ ]:
n_runs = 20
query = "How do neural networks learn?"

timings = {}

# Dense
times = []
for _ in range(n_runs):
    start = time.perf_counter()
    retriever.retrieve(query, top_k=5)
    times.append((time.perf_counter() - start) * 1000)
timings["Dense"] = times

# BM25
times = []
for _ in range(n_runs):
    start = time.perf_counter()
    tokens = query.lower().split()
    bm25.get_scores(tokens)
    times.append((time.perf_counter() - start) * 1000)
timings["BM25"] = times

# Hybrid RRF
times = []
for _ in range(n_runs):
    start = time.perf_counter()
    hybrid.retrieve(query, top_k=5, fusion="rrf")
    times.append((time.perf_counter() - start) * 1000)
timings["Hybrid RRF"] = times

# Dense + Rerank
times = []
for _ in range(n_runs):
    start = time.perf_counter()
    initial = retriever.retrieve(query, top_k=10)
    reranker.rerank(query, initial, top_k=5)
    times.append((time.perf_counter() - start) * 1000)
timings["Dense + Rerank"] = times

fig, ax = plt.subplots(figsize=(10, 5))
labels = list(timings.keys())
medians = [np.median(t) for t in timings.values()]
colors = ["#4C72B0", "#55A868", "#DD8452", "#C44E52"]
bars = ax.barh(labels, medians, color=colors)
for bar, med in zip(bars, medians):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{med:.1f} ms", va="center", fontsize=10)
ax.set_xlabel("Median Latency (ms)")
ax.set_title(f"Retrieval Latency per Query (median of {n_runs} runs)")
plt.tight_layout()
plt.show()

print("\nDetailed stats:")
for method, times in timings.items():
    print(f"  {method:>16}: median={np.median(times):.1f}ms, "
          f"p95={np.percentile(times, 95):.1f}ms, "
          f"mean={np.mean(times):.1f}ms")

## Key Takeaways

- **Dense retrieval** handles semantic/paraphrased queries well but can miss exact keywords, acronyms, and technical codes.
- **BM25** is fast and catches exact keyword matches perfectly, but fails on paraphrased queries with no overlapping terms.
- **Hybrid (RRF)** gets the best of both worlds — it handles both semantic and keyword queries without needing score normalization.
- **Cross-encoder reranking** improves precision at the cost of latency. The pattern "retrieve broadly, rerank narrowly" (e.g., retrieve 20, rerank to 5) is standard in production systems.
- **Latency trade-off**: BM25 is fastest, dense is moderate, reranking adds 100-300ms. Choose based on your quality vs. latency budget.

Next: Step 12 will add the LLM client (Ollama), and Step 13 will build prompt templates to generate answers from retrieved contexts.